# Step 4 — Build ML Dataset

Scans `noel_part/cleaned_data_final/` and builds train / val / test sequence datasets.

- **Split method**: by ICAO24 (prevents same aircraft in both train and test)
- **Output**: `artifacts/step4_ml_dataset/dataset/sequences_{train,val,test}.npz`
- **Also produces**: `artifacts/step4_ml_dataset/catalog/flight_splits.parquet`

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebook' else NOTEBOOK_DIR
sys.path.insert(0, str(NOTEBOOK_DIR))

CLEAN_ROOT    = PROJECT_ROOT / 'noel_part' / 'cleaned_data_final'
STEP4_ROOT    = PROJECT_ROOT / 'artifacts' / 'step4_ml_dataset'
STEP4_CAT     = STEP4_ROOT / 'catalog'
STEP4_DAT     = STEP4_ROOT / 'dataset'

# ── Controls ─────────────────────────────────────────────────────────────────
RUN_BUILD    = True    # set False to just review existing outputs
MAX_FLIGHTS  = None    # None = all flights; set e.g. 200 for a quick preview

print(f'Project root : {PROJECT_ROOT}')
print(f'Clean root   : {CLEAN_ROOT}  exists={CLEAN_ROOT.exists()}')
print(f'Run build    : {RUN_BUILD}')
print(f'Max flights  : {MAX_FLIGHTS}')

In [ ]:
from step4_build_ml_dataset_aeroml3 import Step4Config, run_step4

if RUN_BUILD:
    cfg = Step4Config(
        clean_root=CLEAN_ROOT,
        output_root=STEP4_ROOT,
        max_flights=MAX_FLIGHTS,
        verbose=True,
        progress_every=200,
    )
    summary = run_step4(cfg)
else:
    summary_path = STEP4_CAT / 'step4_summary.json'
    if not summary_path.exists():
        raise FileNotFoundError(f'No Step 4 outputs found at {STEP4_ROOT}. Set RUN_BUILD=True.')
    summary = json.loads(summary_path.read_text())
    print('Loaded existing Step 4 summary.')

print(json.dumps(summary, indent=2))

In [ ]:
# ── Dataset overview ─────────────────────────────────────────────────────────
splits_df = pd.read_parquet(STEP4_CAT / 'flight_splits.parquet')

pd.DataFrame([
    ('total_flights',      summary['total_flights_found']),
    ('failed',            summary['flights_failed']),
    ('train_flights',     summary['split_flights']['train']),
    ('val_flights',       summary['split_flights']['val']),
    ('test_flights',      summary['split_flights']['test']),
    ('train_sequences',   summary['sequence_flights']['train']),
    ('val_sequences',     summary['sequence_flights']['val']),
    ('test_sequences',    summary['sequence_flights']['test']),
], columns=['metric', 'value'])

In [ ]:
# ── Splits by source run ──────────────────────────────────────────────────────
split_by_run = (
    splits_df.groupby(['source_run', 'split'])
    .size().unstack(fill_value=0).reset_index()
)
display(split_by_run)

ax = split_by_run.set_index('source_run')[['train','val','test']].plot(
    kind='bar', figsize=(14, 4), title='Flight splits by source run')
ax.set_ylabel('Flight count')
plt.xticks(rotation=70, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
# ── Inspect one sequence NPZ ──────────────────────────────────────────────────
train_npz = np.load(STEP4_DAT / 'sequences_train.npz', allow_pickle=True)
print('Train NPZ arrays:')
for k, v in train_npz.items():
    print(f'  {k:25s}: {v.shape}  dtype={v.dtype}')

print(f'\nExample segment_id: {train_npz["segment_ids"][0]}')
print(f'Gap duration (s):   {train_npz["gap_dur_sec"][0]:.0f}  ({train_npz["gap_dur_sec"][0]/60:.1f} min)')
print(f'ADS-C waypoints:    {train_npz["n_adsc_wp"][0]}')
print(f'Before anchor:      lat={train_npz["before_anchor_lat"][0]:.3f}  lon={train_npz["before_anchor_lon"][0]:.3f}')
print(f'After  anchor:      lat={train_npz["after_anchor_lat"][0]:.3f}  lon={train_npz["after_anchor_lon"][0]:.3f}')

In [ ]:
# ── Gap duration distribution ─────────────────────────────────────────────────
all_gaps = np.concatenate([
    np.load(STEP4_DAT / f'sequences_{s}.npz', allow_pickle=True)['gap_dur_sec']
    for s in ['train','val','test']
]) / 60.0  # minutes

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(all_gaps, bins=40, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Gap duration (min)'); axes[0].set_ylabel('Flights')
axes[0].set_title('Gap duration distribution')

# ADS-C waypoint count
all_wp = np.concatenate([
    np.load(STEP4_DAT / f'sequences_{s}.npz', allow_pickle=True)['n_adsc_wp']
    for s in ['train','val','test']
])
axes[1].hist(all_wp, bins=30, color='tomato', edgecolor='white')
axes[1].set_xlabel('ADS-C waypoints'); axes[1].set_ylabel('Flights')
axes[1].set_title('ADS-C waypoints per flight')
plt.tight_layout(); plt.show()

print(f'Gap duration: mean={all_gaps.mean():.1f} min  median={np.median(all_gaps):.1f} min')
print(f'ADS-C wp:     mean={all_wp.mean():.1f}  median={np.median(all_wp):.1f}')

In [ ]:
# ── Example flight trajectory ─────────────────────────────────────────────────
import random
train_npz = np.load(STEP4_DAT / 'sequences_train.npz', allow_pickle=True)
i = random.randint(0, len(train_npz['segment_ids'])-1)

seg_id  = str(train_npz['segment_ids'][i])
targets = train_npz['adsc_targets'][i]
mask    = train_npz['adsc_mask'][i]
ba_lat  = float(train_npz['before_anchor_lat'][i])
ba_lon  = float(train_npz['before_anchor_lon'][i])
aa_lat  = float(train_npz['after_anchor_lat'][i])
aa_lon  = float(train_npz['after_anchor_lon'][i])
valid   = mask > 0

plt.figure(figsize=(10, 5))
plt.plot(targets[valid, 1], targets[valid, 0], 'go-', ms=4, label='ADS-C truth')
plt.scatter([ba_lon, aa_lon], [ba_lat, aa_lat],
            s=120, zorder=5, marker='*', color='navy', label='ADS-B anchors')
plt.xlabel('Longitude'); plt.ylabel('Latitude')
plt.title(f'Example flight: {seg_id}')
plt.legend(); plt.tight_layout(); plt.show()
print(f'ADS-C points: {valid.sum()}')
print(f'Gap: {train_npz["gap_dur_sec"][i]/60:.1f} min')